In [ ]:
import os
import sys

import pandas as pd

sys.path.append(os.path.abspath("../"))

In [ ]:
def load_and_process_runs(base_path, prefix):
    runs = [f for f in os.listdir(base_path) if f.startswith(prefix) and f.endswith(".csv")]
    dfs = [pd.read_csv(os.path.join(base_path, f)) for f in runs]
    return dfs


grayscale_path = "../../data/evaluation/cpn-grayscale"
yuyv_path = "../../data/evaluation/cpn-yuyv"

grayscale_runs = load_and_process_runs(grayscale_path, "run_")
yuyv_runs = load_and_process_runs(yuyv_path, "run_")
id_cols = [
    col
    for col in grayscale_runs[0].columns
    if not col.startswith(("classifier", "encoder", "total_loss"))
]

diffs = []
for i in range(min(len(grayscale_runs), len(yuyv_runs))):
    df_grayscale = grayscale_runs[i]
    df_yuyv = yuyv_runs[i]
for col in df_grayscale.columns:
    if col.startswith(("classifier", "encoder")):
        df_grayscale[col] = df_grayscale[col] - df_yuyv[col]
    diffs.append(df_grayscale)

mean_grayscale = pd.concat(grayscale_runs).groupby(id_cols).mean().reset_index()
mean_yuyv = pd.concat(yuyv_runs).groupby(id_cols).mean().reset_index()
mean_diff = pd.concat(diffs).groupby(id_cols).mean().reset_index()
mean_grayscale.to_csv(f"{grayscale_path}/mean.csv", index=False)
mean_yuyv.to_csv(f"{yuyv_path}/mean.csv", index=False)
mean_diff.to_csv("../../data/evaluation/cpn-grayscale-diff-mean.csv", index=False)


### Build Latex Table

In [ ]:
diff_table = (
    mean_yuyv.groupby(["resolution", "architecture"])
    .apply(
        lambda x: f"& CPN$_{x['architecture'].iloc[0].split('_')[-1]}$ & "
        + " & ".join(
            [
                str(x["encoder_recall_at_k_ball"].iloc[0]),
                str(x["encoder_recall_at_k_penaltyMark"].iloc[0]),
                str(x["encoder_recall_at_k_intersections"].iloc[0]),
                str(x["encoder_mae_ball"].iloc[0]),
                str(x["encoder_mae_penaltyMark"].iloc[0]),
                str(x["encoder_mae_intersections"].iloc[0]),
            ]
        )
        + " \\\\"
    )
    .reset_index()
)

diff_table.columns = ["resolution", "architecture", "formatted_string"]

for resolution in sorted(diff_table["resolution"].unique(), reverse=True):
    print(resolution)
    print(
        "\n".join(diff_table[diff_table["resolution"] == resolution]["formatted_string"].tolist())
    )
